# Reset - put everything back to the empty starting state

**Why this exists.** The demo only impresses if the audience sees the change happen: empty tables filling up, a new column appearing, a pipeline failing and recovering. After a run-through, those changes are permanent. This notebook undoes them.

**When to run it.**
1. After every rehearsal.
2. Once more shortly before the live call.

**This is not part of the demo.** Never run it in front of the customer.

**Run every cell from top to bottom, then do the UI step at the end.**

### RUN DURING DEMO
Set the role, warehouse and schema.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE WAREHOUSE WH_INGEST;
USE SCHEMA BRONZE;

### RUN DURING DEMO
Empty the three Bronze tables.

In [ ]:
%%sql -r truncate_bronze
TRUNCATE TABLE MEMBERS_RAW;
TRUNCATE TABLE CLAIMS_RAW;
TRUNCATE TABLE LABS_RAW;

### RUN DURING DEMO
Demo 1 (Concurrency): empty the daily-load table, which grows by about 30 million rows every rehearsal, and put the four demo warehouses back to their demo settings (all Medium, WH_BI 1 to 4 clusters, auto-suspend 20 minutes).

In [ ]:
%%sql -r reset_demo1
TRUNCATE TABLE SOMATUS_DEMO.BRONZE.CLAIMS_DAILY_LOAD;
ALTER WAREHOUSE WH_SHARED_SYNAPSE_SIM SET WAREHOUSE_SIZE = 'MEDIUM' MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 1 AUTO_SUSPEND = 1200;
ALTER WAREHOUSE WH_INGEST             SET WAREHOUSE_SIZE = 'MEDIUM' MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 1 AUTO_SUSPEND = 1200;
ALTER WAREHOUSE WH_TRANSFORM          SET WAREHOUSE_SIZE = 'MEDIUM' MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 1 AUTO_SUSPEND = 1200;
ALTER WAREHOUSE WH_BI                 SET WAREHOUSE_SIZE = 'MEDIUM' MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 4 AUTO_SUSPEND = 1200;

### RUN DURING DEMO
Recreate Silver and Gold without the new columns. This must happen before the next cell, otherwise dropping the columns would break them.

In [ ]:
%%sql -r baseline_dts
CREATE OR REPLACE DYNAMIC TABLE SOMATUS_DEMO.SILVER.MEMBERS_CLEAN
  TARGET_LAG='1 minute' WAREHOUSE=WH_TRANSFORM REFRESH_MODE=INCREMENTAL AS
SELECT MEMBER_ID, CLIENT_ID, DOB, GENDER, CKD_STAGE, STATE, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.BRONZE.MEMBERS_RAW
WHERE MEMBER_ID IS NOT NULL AND DOB IS NOT NULL AND LENGTH(STATE) = 2
QUALIFY ROW_NUMBER() OVER (PARTITION BY MEMBER_ID ORDER BY LOADED_AT DESC) = 1;
CREATE OR REPLACE DYNAMIC TABLE SOMATUS_DEMO.SILVER.CLIENT_CLAIMS
  TARGET_LAG='1 minute' WAREHOUSE=WH_TRANSFORM REFRESH_MODE=INCREMENTAL AS
SELECT CLAIM_ID, MEMBER_ID, CLIENT_ID, TRY_TO_DATE(SERVICE_DATE) AS SERVICE_DATE, CPT_CODE,
       PAID_AMOUNT::NUMBER(12,2) AS PAID_AMOUNT, CLAIM_STATUS, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.BRONZE.CLAIMS_RAW;
CREATE OR REPLACE DYNAMIC TABLE SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY
  TARGET_LAG='1 minute' WAREHOUSE=WH_TRANSFORM REFRESH_MODE=INCREMENTAL AS
SELECT CLIENT_ID, CKD_STAGE, COUNT(*) AS MEMBERS
FROM SOMATUS_DEMO.SILVER.MEMBERS_CLEAN GROUP BY 1, 2;

### RUN DURING DEMO
Remove the columns that schema evolution added, so it can be shown again live.

In [ ]:
%%sql -r drop_evolved
ALTER TABLE MEMBERS_RAW DROP COLUMN IF EXISTS DIALYSIS_MODALITY;
ALTER TABLE CLAIMS_RAW  DROP COLUMN IF EXISTS PLACE_OF_SERVICE;

### RUN DURING DEMO
Recreate the pipes. This wipes their memory of which files they already loaded, so the same files load again next time. The queue address (and therefore the S3 notification) stays the same.

In [ ]:
%%sql -r recreate_pipes
CREATE OR REPLACE PIPE PIPE_MEMBERS AUTO_INGEST=TRUE AS
COPY INTO MEMBERS_RAW FROM @S3_CLIENT_DROP/members/
  FILE_FORMAT=(FORMAT_NAME=FF_CSV) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE
  INCLUDE_METADATA=(FILE_NAME=METADATA$FILENAME, LOADED_AT=METADATA$START_SCAN_TIME) ON_ERROR=CONTINUE;
CREATE OR REPLACE PIPE PIPE_CLAIMS AUTO_INGEST=TRUE AS
COPY INTO CLAIMS_RAW FROM @S3_CLIENT_DROP/claims/
  FILE_FORMAT=(FORMAT_NAME=FF_AVRO) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE
  INCLUDE_METADATA=(FILE_NAME=METADATA$FILENAME, LOADED_AT=METADATA$START_SCAN_TIME) ON_ERROR=CONTINUE;
CREATE OR REPLACE PIPE PIPE_LABS AUTO_INGEST=TRUE AS
COPY INTO LABS_RAW (RAW, FILE_NAME, LOADED_AT)
  FROM (SELECT $1, METADATA$FILENAME, METADATA$START_SCAN_TIME FROM @S3_CLIENT_DROP/labs/)
  FILE_FORMAT=(FORMAT_NAME=FF_JSON) ON_ERROR=CONTINUE;

### RUN DURING DEMO
Turn the failure switch off, clear the pipeline log, empty the concurrency demo table, and restore WH_BI to 4 clusters.

In [ ]:
%%sql -r reset_ops
UPDATE SOMATUS_DEMO.OPS.DEMO_CONTROL SET VAL = 'FALSE' WHERE KEY = 'FAIL_VALIDATE';
TRUNCATE TABLE SOMATUS_DEMO.OPS.PIPELINE_LOG;
TRUNCATE TABLE SOMATUS_DEMO.BRONZE.CLAIMS_DAILY_LOAD;
ALTER WAREHOUSE WH_BI SET MAX_CLUSTER_COUNT = 4;
TRUNCATE TABLE SOMATUS_DEMO.OPS.LOAD_ERROR_LOG;            -- so the CIGNA file emails again
REMOVE @SOMATUS_DEMO.OPS.LOAD_ERROR_EXPORTS;               -- old rejected-row CSVs


### RUN DURING DEMO
Demo 07 (Time Travel and clone): drop the demo table and its clone. Notebook 07 recreates them.

In [ ]:
%%sql -r reset_demo7
DROP TABLE IF EXISTS SOMATUS_DEMO.GOLD.CLAIMS_SNAPSHOT_DEV;
DROP TABLE IF EXISTS SOMATUS_DEMO.GOLD.CLAIMS_SNAPSHOT;

### RUN DURING DEMO
Refresh the dynamic tables so they reflect the now-empty Bronze.

In [ ]:
%%sql -r refresh_dts
ALTER DYNAMIC TABLE SOMATUS_DEMO.SILVER.MEMBERS_CLEAN REFRESH;
ALTER DYNAMIC TABLE SOMATUS_DEMO.SILVER.LAB_RESULTS REFRESH;
ALTER DYNAMIC TABLE SOMATUS_DEMO.SILVER.CLIENT_CLAIMS REFRESH;
CREATE OR REPLACE DYNAMIC TABLE SOMATUS_DEMO.GOLD.CLAIMS_PUBLISHED
  TARGET_LAG = 'DOWNSTREAM' WAREHOUSE = WH_TRANSFORM REFRESH_MODE = INCREMENTAL INITIALIZE = ON_CREATE AS
SELECT CLAIM_ID, MEMBER_ID, CLIENT_ID, SERVICE_DATE, CPT_CODE, PAID_AMOUNT, CLAIM_STATUS, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.SILVER.CLIENT_CLAIMS;

### RUN DURING DEMO
Check: every count should be 0.

In [ ]:
%%sql -r verify
SELECT 'MEMBERS_RAW' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM MEMBERS_RAW UNION ALL
SELECT 'CLAIMS_RAW', COUNT(*) FROM CLAIMS_RAW UNION ALL
SELECT 'LABS_RAW',   COUNT(*) FROM LABS_RAW UNION ALL
SELECT 'CLAIMS_DAILY_LOAD', COUNT(*) FROM SOMATUS_DEMO.BRONZE.CLAIMS_DAILY_LOAD UNION ALL
SELECT 'MEMBERS_CLEAN', COUNT(*) FROM SOMATUS_DEMO.SILVER.MEMBERS_CLEAN;

### UI STEP - do this in the screen, not in the notebook
**Empty the S3 folders, but keep the folders**

1. Open the AWS console and go to **S3 > mgupta-bucket > soybase > somatus**.
2. Open **members**. Tick the box next to each file (not the folder itself). Click **Delete**, type **permanently delete**, and confirm.
3. Do the same inside **claims** and inside **labs**.
4. Do not delete the folders. If you delete and recreate a folder, S3 creates an empty placeholder object that shows up as a blank 0-row load in Copy history.

### DEMO DAY ONLY - read before you uncomment
**Turn on the automatic bad-file email.** The alert `ALERT_LOAD_ERRORS` stays suspended between rehearsals so it doesn't run every minute. On demo day, after everything above is done:
1. Uncomment the `RESUME` line and run the cell. The alert then checks every minute, and the bad file in Pass 3 sends its email by itself in about 1-2 minutes.
2. Check the result shows `started`.
3. After the call, uncomment the `SUSPEND` line and run it again (comment `RESUME` back out).
Backup during the demo: if the email hasn't arrived, run `fire_alert` in 00.

In [ ]:
%%sql -r alert_on
-- ALTER ALERT SOMATUS_DEMO.OPS.ALERT_LOAD_ERRORS RESUME;     -- demo day: before the call
-- ALTER ALERT SOMATUS_DEMO.OPS.ALERT_LOAD_ERRORS SUSPEND;    -- after the call
SHOW ALERTS LIKE 'ALERT_LOAD_ERRORS' IN SCHEMA SOMATUS_DEMO.OPS
->> SELECT "name" AS ALERT_NAME, "state" AS STATE, "schedule" AS SCHEDULE FROM $1;

## Fallback if S3 notifications fail during the live call
Load the same files from the internal stage instead. The behaviour is identical; it is just not triggered by an event.
```sql
COPY INTO SOMATUS_DEMO.BRONZE.MEMBERS_RAW FROM @SOMATUS_DEMO.BRONZE.INT_UPLOAD PATTERN='.*members_.*'
  FILE_FORMAT=(FORMAT_NAME=SOMATUS_DEMO.BRONZE.FF_CSV) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE
  INCLUDE_METADATA=(FILE_NAME=METADATA$FILENAME, LOADED_AT=METADATA$START_SCAN_TIME) ON_ERROR=CONTINUE;
```